# Tests 

In [99]:
from autograd.engine import backward, zero_grad
from nn import *

## Experiment 1 — Increase Depth

Changing architecture from:

2 → 3 → 1

To:

2 → 5 → 5 → 5 → 1


### Layers
2 → 5 → 5 → 5 → 1

In [100]:
hidden_layer = layer(2,5)
h2_layer = layer(5,5)
h3_layer = layer(5,5)
output_layer = layer(5,1)

In [101]:
dataset = [
    (2,3,16),
    (1,1,5)
]

In [ ]:
n = 0.001
for step in range(101):
    for x1, x2, ytrue in dataset:
        x = [x1, x2]

        for neuron in hidden_layer.neurons:
            for w in neuron.w:
                zero_grad(w)
            zero_grad(neuron.b)

        for neuron in h2_layer.neurons:
            for w in neuron.w:
                zero_grad(w)
            zero_grad(neuron.b)

        for neuron in h3_layer.neurons:
            for w in neuron.w:
                zero_grad(w)
            zero_grad(neuron.b)

        for neuron in output_layer.neurons:
            for w in neuron.w:
                zero_grad(w)
            zero_grad(neuron.b)

        h1_nodes = hidden_layer.forward(x)
        a1 = [relu(node) for node in h1_nodes]

        h2_nodes = h2_layer.forward(a1)
        a2 = [relu(node) for node in h2_nodes]

        h3_nodes = h3_layer.forward(a2)
        a3 = [relu(node) for node in h3_nodes]

        y = output_layer.forward(a3)[0]

        l = (y - ytrue)**2

        backward(l)

        for neuron in hidden_layer.neurons:
            print([w.grad for w in neuron.w], neuron.b.grad)

        for neuron in h2_layer.neurons:
            print([w.grad for w in neuron.w], neuron.b.grad)

        for neuron in h3_layer.neurons:
            print([w.grad for w in neuron.w], neuron.b.grad)

        for neuron in hidden_layer.neurons:
            for w in neuron.w:
                w.value -= n * w.grad
            neuron.b.value -= n * neuron.b.grad

        for neuron in h2_layer.neurons:
            for w in neuron.w:
                w.value -= n * w.grad
            neuron.b.value -= n * neuron.b.grad

        for neuron in h3_layer.neurons:
            for w in neuron.w:
                w.value -= n * w.grad
            neuron.b.value -= n * neuron.b.grad

        for neuron in output_layer.neurons:
            for w in neuron.w:
                w.value -= n * w.grad
            neuron.b.value -= n * neuron.b.grad

    if step % 10 == 0:
        print("Step:", step, "| Loss:", l.value)

### Engineering Insight Depth & Gradient Stability

Increasing the architecture from 2 → 3 → 1 to 2 → 5 → 5 → 5 → 1 exposed fundamental optimization challenges in deep networks.

While the shallow network converged rapidly to near-zero loss, the deeper network:
- Converged significantly slower
- Required a smaller learning rate
- Exhibited near-zero gradients in early layers
- Produced multiple dead ReLU neurons

Why This Happens
Backpropagation propagates gradients through repeated multiplication:

dL/dw1 = dL/dw4 * w4 * ReLU' * w3 * ReLU' * w2 * ReLU'

Each additional layer introduces more multiplicative terms.

If:
- |W| < 1 → gradients shrink exponentially (vanishing gradients)
- ReLU' = 0 → gradients are completely blocked
- Learning rate is too high → updates become unstable

Depth amplifies instability.

Observed Behaviors
- Early-layer gradients were significantly smaller than output-layer gradients.
- Many neurons had zero gradients due to ReLU gating.
- Loss required a reduced learning rate to remain stable.
- Convergence plateaued compared to the shallow network.

Key Takeaway

- Deep networks are not inherently stable.

Building this from scratch revealed that backpropagation is highly sensitive to depth, scale, and activation behavior.

## Experiment 2 - Learning Rate Instability

Change:
lr = 0.001

To:
lr = 1.0

In [103]:
n = 1.0
for step in range(101):
    for x1, x2, ytrue in dataset:
        x = [x1, x2]

        for neuron in hidden_layer.neurons:
            for w in neuron.w:
                zero_grad(w)
            zero_grad(neuron.b)

        for neuron in h2_layer.neurons:
            for w in neuron.w:
                zero_grad(w)
            zero_grad(neuron.b)

        for neuron in h3_layer.neurons:
            for w in neuron.w:
                zero_grad(w)
            zero_grad(neuron.b)

        for neuron in output_layer.neurons:
            for w in neuron.w:
                zero_grad(w)
            zero_grad(neuron.b)

        h1_nodes = hidden_layer.forward(x)
        a1 = [relu(node) for node in h1_nodes]

        h2_nodes = h2_layer.forward(a1)
        a2 = [relu(node) for node in h2_nodes]

        h3_nodes = h3_layer.forward(a2)
        a3 = [relu(node) for node in h3_nodes]

        y = output_layer.forward(a3)[0]

        l = (y - ytrue)**2

        backward(l)

        for neuron in hidden_layer.neurons:
            print([w.grad for w in neuron.w], neuron.b.grad)

        for neuron in h2_layer.neurons:
            print([w.grad for w in neuron.w], neuron.b.grad)

        for neuron in h3_layer.neurons:
            print([w.grad for w in neuron.w], neuron.b.grad)

        for neuron in hidden_layer.neurons:
            for w in neuron.w:
                w.value -= n * w.grad
            neuron.b.value -= n * neuron.b.grad

        for neuron in h2_layer.neurons:
            for w in neuron.w:
                w.value -= n * w.grad
            neuron.b.value -= n * neuron.b.grad

        for neuron in h3_layer.neurons:
            for w in neuron.w:
                w.value -= n * w.grad
            neuron.b.value -= n * neuron.b.grad

        for neuron in output_layer.neurons:
            for w in neuron.w:
                w.value -= n * w.grad
            neuron.b.value -= n * neuron.b.grad

    if step % 10 == 0:
        print("Step:", step, "| Loss:", l.value)

[0, 0] 0
[-0.014979624983644906, -0.02246943747546736] -0.007489812491822453
[-0.0645217059037206, -0.0967825588555809] -0.0322608529518603
[0, 0] 0
[-0.008700266017156469, -0.013050399025734703] -0.004350133008578234
[0.0, -0.08700157301503687, -0.06071513964278515, 0.0, -0.07492114608832383] -0.35403343712594526
[0.0, -0.0724788843468203, -0.05058029908849924, 0.0, -0.06241497589392234] -0.2949366046510876
[0, 0.0, 0.0, 0, 0.0] 0
[0, 0.0, 0.0, 0, 0.0] 0
[0, 0.0, 0.0, 0, 0.0] 0
[0.023730893398992463, 0.041558877000576845, 0.0, 0.0, 0.0] 1.3324691815286591
[-0.2270871648135571, -0.39768783215287745, 0.0, 0.0, 0.0] -12.75074829874003
[0.0, 0.0, 0, 0, 0] 0
[0.0, 0.0, 0, 0, 0] 0
[0.0, 0.0, 0, 0, 0] 0
[0, 0] 0
[403.3924931953389, 403.3924931953389] 403.3924931953389
[381.42741659524427, 381.42741659524427] 381.42741659524427
[0, 0] 0
[214.30783283592663, 214.30783283592663] 214.30783283592663
[0.0, 157.17804621886432, 337.7783906926931, 0.0, 122.27492763131296] 1330.9819239456817
[0.0, 272

### Engineering Insight — Learning Rate Instability

Increasing the learning rate from 0.001 to 1.0 exposed fundamental optimization instability in gradient descent.

- While the smaller learning rate allowed gradual convergence, the larger learning rate:
- Caused the loss to increase instead of decrease
- Produced oscillating behavior across steps
- Led to unstable parameter updates
- Prevented convergence entirely

Why This Happens

Gradient descent updates parameters using:

𝑊=𝑊−𝜂⋅∇𝐿

Where:

- η is the learning rate
- ∇L is the gradient

With lr = 1.0, updates become extremely large:

wnew = wold−1.0*grad

If gradients are moderately large, the parameter jump overshoots the minimum.

Instead of descending smoothly along the loss surface:
- Small learning rate → stable descent
- Large learning rate → overshooting
- Very large learning rate → divergence

Optimization becomes unstable.

Why Depth Makes This Worse

In deeper networks, gradients are products of multiple terms:

dL/dw1 = dL/dw4 * w4 * ReLU′ * w3 * ReLU′ * w2 * ReLU′

Even small parameter changes in early layers can amplify through depth.

If:
- Gradients are large → updates explode
- Learning rate is large → instability compounds
- Multiple layers amplify error → divergence accelerates

Depth magnifies step-size sensitivity.

Observed Behaviors

- Loss increased instead of decreasing
- Training oscillated between high values
- Gradients grew larger across steps
- Network failed to converge

Key Takeaway

- A correct gradient is not sufficient for stable learning.
- Optimization stability depends heavily on step size.

Building this from scratch revealed that training deep networks requires careful control over update magnitude.